In case that you use data from this Notebook as it is originally provided (network `3X2023`), do not forget to use the proper citation for attribution:

*   Afonso Loureiro (2024): GeoLab. GFZ Data Services. Dataset/DAS. doi:[10.14470/8K802502](https://doi.org/10.14470/8K802502)

DISCLAIMER: This example notebook is provided 'as is', with no warranty of accuracy, completeness, or fitness for any purpose. Use at your own risk.

`boto3` is one of the libraries you could use from python to access data archived in some S3 Object Storage.

In [ ]:
import os
from datetime import datetime

import boto3
import numpy as np
from botocore import UNSIGNED
from botocore.config import Config
from h5py import File
from matplotlib import pyplot as plt

It is very important to define who is actually hosting the data. Most of the examples you will find will be hosted in Amazon, but your Universities could also be providers of the S3 service.

In [ ]:
endpoint = "https://s3.gfz.de/"

Pay attention to the `UNSIGNED` value in the configuration. In this particular case, you will access data that does not require to authenticate. This is not so usual in the S3 world. Usually you will be given a pair of keys (credentials) to operate with the service provider.

Within `boto3` there are 2 ways to operate on the buckets and objects. Through a `resource` or a `client`. In this examlpe below we use the `resource` to get the data.

In [ ]:
s3r = boto3.resource(
    "s3", endpoint_url=endpoint, config=Config(signature_version=UNSIGNED)
)

Once the `resource` is created you can define the objects to operate with. For instance, the `Bucket` where the 3X2023 data (GeoLab - Madeira) is hosted.

In [ ]:
bucket = s3r.Bucket("gc.3x2023")

Let's define the time window and the channel to display.

In [ ]:
channel = 500
starttime = datetime(2023, 11, 2, 9, 31, 0)
endtime = datetime(2023, 11, 2, 9, 31, 10)

Create auxiliary variables to loop through objects and keep the ones that lie within the time window. Here, we will check all objects that were created in the hour of the time window.

In [ ]:
prefix = starttime.strftime("%Y%m%d/dphi/%H")
h5start = starttime.strftime("%Y%m%d/dphi/%H%M%S.h5")
h5end = endtime.strftime("%Y%m%d/dphi/%H%M%S.h5")
files = list()
prevfile = None

Loop throug `objects` and always add a previous file to keep the beginning of the time window. Stop as soon as the time window is gone. After the loop, add the last file that is missing.

In [ ]:
for obj in bucket.objects.filter(Prefix=prefix):
    # If time window is already gone then break
    if h5end < obj.key:
        break
    # If we are in the time window
    if h5start <= obj.key <= h5end:
        print(prevfile)
        files.append(prevfile)
    prevfile = obj.key

# If there was a file still pending
if prevfile is not None:
    files.append(prevfile)
    print(prevfile)

Use the `download_file` method to get some object(s) and save it physically in your local computer. Then, open it and extract the channel you want to display.

In [ ]:
stream = np.array([])
for f in files:
    _, fname = os.path.split(f)
    bucket.download_file(Key=f, Filename=fname)
    fi = File(fname)
    stream = np.append(stream, fi["data"][:, channel])

Check what you got...

In [ ]:
plt.plot(stream)
plt.show()